## Preparation

In [ ]:
# system imports
import os
from pathlib import Path
from dotenv import load_dotenv

# math
import numpy as np

# pdf readers
from pypdf import PdfReader
import pymupdf4llm

# RAG-related
import tiktoken
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# loading env
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
# path to pdf
FILEPATH = 'data/transformers.pdf'

## Data extraction

In [ ]:
def extract_text(pdf_file_path: str) -> str:
    """A function to extract text from a PDF file."""
    reader = PdfReader(pdf_file_path)
    return "\n\n".join(doc.extract_text() or "" for doc in reader.pages)

In [ ]:
extracted_pdf = extract_text(FILEPATH)

In [ ]:
def extract_markdown(pdf_file_path: str, save_md_path: str = None) -> str:
    """A function to extract llm-ready markdown from a PDF file."""
    md_text = pymupdf4llm.to_markdown(pdf_file_path)

    if save_md_path is not None:
        filename = Path(pdf_file_path).stem
        Path(f"{save_md_path}/{filename}.md").write_text(md_text, encoding="utf-8")

    return md_text

In [ ]:
extracted_markdown = extract_markdown(FILEPATH, "data")

## Splitting texts

### Calculating the number of tokens per chunk 

In [ ]:
# count tokens considering ~4 char / token
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [ ]:
def analyze_paragraphs(text_to_analyze: str) -> dict:
    """A function to analyze the length of paragraphs."""
    paragraphs = text_to_analyze.split("\n\n")

    all_lengths = [
        num_tokens_from_string(p.strip(), "cl100k_base")
        for p in paragraphs if p.strip()
    ]

    filtered_lengths = [length for length in all_lengths if length > 15]

    median_length, ninety_percentile = np.percentile(filtered_lengths, [50, 90])

    result_dict = {
        "num_paragraphs_before_filtering": len(all_lengths),
        "num_paragraphs_after_filtering": len(filtered_lengths),
        "median_length": float(median_length),
        "ninety_percentile": float(ninety_percentile),
    }

    return result_dict

In [ ]:
analyze_dict = analyze_paragraphs(extracted_markdown)
analyze_dict

130 token length per paragraph -> optimal number of tokens per chunk, with ceiling 130 to 150 -> $150*3=450$

### Two-Splitter approach
First to split by headers, then - by tokens.

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

In [ ]:
# header splitter
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)
md_splits = md_splitter.split_text(extracted_markdown)

In [ ]:
# text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=350,
    chunk_overlap=45,
)
final_splits = text_splitter.split_documents(md_splits)

## Embedding

In [ ]:
# embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    api_key=GEMINI_API_KEY,
)

In [ ]:
# creating and storing embeddings locally
vectorstore = Chroma.from_documents(
    documents=final_splits,
    embedding=embeddings,
    persist_directory="data"
)

In [ ]:
retriever = vectorstore.as_retriever()

## Generation

In [ ]:
# post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \nContext: {context} \nAnswer:"
)

In [ ]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [ ]:
# chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# question
rag_chain.invoke("What is Transformer?")